Dummy Classifier (Baseline)

This notebook implements a baseline model that makes predictions using simple strategies (like always picking the most frequent class). This serves as a point of comparison for all other models.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from metrics_utils import calculate_business_metrics


mlflow.set_experiment("Walmart-Sales-Classification")

In [ ]:
train_df = pd.read_csv('../../data/model_ready/train.csv')
test_df = pd.read_csv('../../data/model_ready/test.csv')

features_selected = [
    "Size", "Store", "Dept", "CPI", "DeptFrequency", 
    "Week_cos", "IsPreHoliday", "Week_sin", "Fuel_Price", 
    "ConsumerConfRatio", "AvgMarkDownAmount"
]
target = "Sales_Class"
holiday_col = "IsHoliday"

X_train = train_df[features_selected]
y_train = train_df[target]
X_test = test_df[features_selected]
y_test = test_df[target]

# We need IsHoliday for business metrics evaluation
test_holidays = test_df[holiday_col]

In [ ]:
with mlflow.start_run(run_name="Dummy_Baseline"):
    model_path = 'dummy_model.pkl'
    if os.path.exists(model_path):
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        print('Model loaded from pickle')
    else:
        # 1. Initialize and Train
        model = DummyClassifier(strategy="stratified", random_state=42)
        model.fit(X_train, y_train)
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        print('Model trained and saved to pickle')


In [ ]:

y_pred = model.predict(X_test)


In [ ]:


acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")


In [ ]:
# 4. Calculate Business Metrics
biz_metrics = calculate_business_metrics(y_test, y_pred, test_holidays)

In [ ]:


mlflow.log_param("model_type", "DummyClassifier")
mlflow.log_param("strategy", "stratified")

mlflow.log_metric("accuracy", acc)
mlflow.log_metric("f1_weighted", f1)
mlflow.log_metric("holiday_accuracy", biz_metrics["holiday_accuracy"])
mlflow.log_metric("weighted_classification_error", biz_metrics["weighted_classification_error"])


In [ ]:

    # 6. Save Model Artifact
mlflow.log_artifact(model_path)

print(f"Baseline Accuracy: {acc:.4f}")
print(f"Holiday Accuracy: {biz_metrics['holiday_accuracy']:.4f}")
print(f"Weighted Error: {biz_metrics['weighted_classification_error']:.4f}")
